In [1]:
import sqlite3
from src.config import DB_PATH

start_ts = "2026-09-21T08:12:10.627529+00:00"
end_ts = "2026-09-21T08:16:43.809884+00:00"

conn = sqlite3.connect(DB_PATH, timeout=10)
try:
    metrics_ids = [row[0] for row in conn.execute(
        "SELECT id FROM metrics WHERE timestamp BETWEEN ? AND ?", (start_ts, end_ts)
    ).fetchall()]

    label_id = conn.execute(
        "INSERT INTO labels (start_ts, end_ts, label, notes) VALUES (?, ?, ?, ?)",
        (start_ts, end_ts, "cpu_stress", "natural capture, Week 4, real un-injected stress test")
    ).lastrowid

    conn.executemany(
        "INSERT INTO label_metrics (label_id, metrics_id) VALUES (?, ?)",
        [(label_id, mid) for mid in metrics_ids]
    )
    conn.commit()
    print(f"labeled {len(metrics_ids)} rows under label_id {label_id}")
finally:
    conn.close()

labeled 8 rows under label_id 11


In [2]:
# from a terminal: python -m src.detectors

from src.detectors import load_joined_data
from src.evaluate import load_latest_run_detections
from src.config import DB_PATH
import pandas as pd

df = load_joined_data(DB_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
session = df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)]

for method in ["cusum", "ocsvm"]:
    detections = load_latest_run_detections(DB_PATH, method)
    flagged = detections[detections["metrics_id"].isin(session["metrics_id"]) & detections["is_anomaly"]]
    print(f"{method}: {len(flagged)}/{len(session)} rows in this session flagged")

cusum: 7/8 rows in this session flagged
ocsvm: 0/8 rows in this session flagged


In [3]:
print("start_ts:", start_ts)
print("end_ts:", end_ts)
print("df timestamp dtype:", df["timestamp"].dtype)
print("df timestamp range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("rows matching window:", df[(df["timestamp"] >= start_ts) & (df["timestamp"] <= end_ts)].shape[0])

start_ts: 2026-09-21T08:12:10.627529+00:00
end_ts: 2026-09-21T08:16:43.809884+00:00
df timestamp dtype: datetime64[us, UTC]
df timestamp range: 2026-09-02 19:16:20.102694+00:00 to 2026-09-21 08:55:24.411951+00:00
rows matching window: 8


In [4]:
conn = sqlite3.connect(DB_PATH)
conn.execute("DELETE FROM label_metrics WHERE label_id = 10")
conn.execute("DELETE FROM labels WHERE id = 10")
conn.commit()
conn.close()

In [5]:
((15.5*4)+(18.66*17)+(16.24*20)+(17.5*6)+(16.06*24)+(17.1*20))/91

16.884175824175824

In [6]:
16.884175824175824/20*4

3.376835164835165